# AI Arena — PPO with Curriculum Learning + BC Warm-Start

Trains the AshGrid combat NN on Kaggle T4 GPU. Improvements over previous
version:

1. **Behavior cloning warm-start** — supervised pretrain from GA-best
   for ~3 minutes. Skips the "learn what fire button does" 5M-step phase.
2. **4-stage curriculum** — small map / close spawn / static opponents
   first, gradually expand to deployment scale (1200×1200).
3. **Reward shaping** decays to zero by stage 4 — final policy uses
   pure kill/death signal but learns from dense feedback early.
4. **Bigger budget** — 16M steps default. Run on Kaggle GPU,
   ~3 hours wall time.

## CONFIG

Edit values then **Run All**.

In [ ]:
# === Total training budget ===
TOTAL_STEPS         = 16_000_000   # PPO env steps (post-BC)
BC_ROLLOUT_STEPS    = 200_000      # GA-best self-play rollout for behavior cloning
BC_EPOCHS           = 50           # supervised epochs over BC dataset

# === Smoke-test mode: set True for a 5-min end-to-end check ===
SMOKE_TEST          = False
if SMOKE_TEST:
    TOTAL_STEPS         = 100_000
    BC_ROLLOUT_STEPS    = 20_000
    BC_EPOCHS           = 10

# === Self-play / curriculum ===
N_ENVS              = 8            # parallel envs (Kaggle GPU has plenty of RAM)
CHECKPOINT_EVERY    = 1_000_000    # save .zip every N steps
FROZEN_OPP_EVERY    = 500_000      # add current policy to self-play pool every N steps
SELF_POOL_MAX       = 8            # cap on frozen self snapshots

# === PPO hyperparams ===
LR_INIT             = 3e-4
ENT_COEF_INIT       = 0.02         # high exploration early
ENT_COEF_FINAL      = 0.005        # tighter exploitation late
N_EPOCHS            = 10
N_STEPS             = 2048
BATCH_SIZE          = 256
GAMMA               = 0.995
GAE_LAMBDA          = 0.95
CLIP_RANGE          = 0.2

# === Output ===
OUTPUT_DIR          = '/kaggle/working/ppo_ckpt'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'TOTAL_STEPS={TOTAL_STEPS:,}')
print(f'BC_ROLLOUT_STEPS={BC_ROLLOUT_STEPS:,}')
print(f'N_ENVS={N_ENVS}, output={OUTPUT_DIR}')

## Install dependencies

In [ ]:
%pip install -q stable-baselines3 onnx onnxruntime

## Load combat_env and kaggle_train

Upload `combat_env.py` and `kaggle_train.py` together as a Kaggle dataset
(any name, e.g. `ai-arena-files`).

In [ ]:
import sys, os

candidates = [
    '/kaggle/input/ai-arena-files',
    '/kaggle/input/ai-arena',
    '/kaggle/input/combat-env',
]
for c in candidates:
    if os.path.exists(os.path.join(c, 'combat_env.py')):
        if c not in sys.path:
            sys.path.insert(0, c)
        print(f'Found combat_env.py at {c}')
        break
else:
    # Recursive search
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'combat_env.py' in files:
            sys.path.insert(0, root)
            print(f'Found combat_env.py at {root}')
            break

import combat_env
from combat_env import (
    CombatEnv, SinglePerspectiveEnv, Curriculum, curriculum_for_step,
    OBS_DIM, ACTION_DIM, SQUAD_SIZE,
    random_opponent, idle_opponent, runner_opponent, make_ga_opponent,
    make_curriculum_opponent_picker,
)
print(f'combat_env loaded. OBS_DIM={OBS_DIM}, ACTION_DIM={ACTION_DIM}')

# Smoke check the curriculum schedule
for frac in (0.05, 0.30, 0.60, 0.90):
    c = curriculum_for_step(int(frac * TOTAL_STEPS), TOTAL_STEPS)
    print(f'  step {int(frac*TOTAL_STEPS):>10,}: {c.world_w}x{c.world_h} '
          f'spawn={c.spawn_dist:.0f} stat={c.p_static:.2f} ga={c.p_ga:.2f} '
          f'self={c.p_self:.2f} vis={c.coef_visibility:.3f}')

## GA-best genome (used as BC teacher + curriculum opponent)

These are values from a 200-generation GA training run. They make the
GA-best opponent a competent baseline — a useful tutor during the early
curriculum stages and a noisy demonstrator for behavior cloning.

In [ ]:
# Reasonable hand-tuned genome (close to GA-best after 200 gens)
DEFAULT_GA_GENOME = {
    'view_arc':            2.4,    # ~140°
    'view_range':          720,
    'snap_to_threat':      0.27,
    'patrol_scan_speed':   0.04,
    'engage_distance':     280,
    'spread':              0.07,
    'fire_cd_frames':      12,
    'cover_dmg_threshold': 28,
    'peek_duration':       55,
    'hide_duration':       40,
    'flee_hp_pct':         0.34,
    'flank_chance':        0.6,
}

ga_policy = make_ga_opponent(DEFAULT_GA_GENOME)
print('GA-best policy ready.')

## Behavior cloning warm-start

Roll out **GA-best vs GA-best** for `BC_ROLLOUT_STEPS` ticks, recording
every `(observation, action)` pair the GA agents produce. Then train a
small MLP to imitate those actions. The resulting weights become PPO's
initialization — saving roughly 5M steps of "discover the fire button".

In [ ]:
import random
import numpy as np
from combat_env import CombatEnv, MOVE_DIRS, _ga_decide_action

print(f'Collecting {BC_ROLLOUT_STEPS:,} (obs, action) pairs from GA-best vs GA-best...')

# Use a stage-3 curriculum for BC so observation distribution is close to
# what the policy will see in the back half of PPO training.
bc_curriculum = curriculum_for_step(int(0.55 * TOTAL_STEPS), TOTAL_STEPS)
bc_curriculum.p_static = 0.0
bc_curriculum.p_runner = 0.0
bc_curriculum.p_random = 0.0
bc_curriculum.p_ga = 1.0   # GA-best opponents
bc_curriculum.p_self = 0.0

bc_obs = []
bc_act = []

env = CombatEnv(opponent_policy=ga_policy, curriculum=bc_curriculum, seed=12345)
obs_list = env.reset(seed=12345)
collected = 0
import time; t0 = time.time()
while collected < BC_ROLLOUT_STEPS:
    # Both teams driven by GA-best; record team-0's view + actions
    actions = [_ga_decide_action(env.units[i], env, DEFAULT_GA_GENOME)
               for i in range(SQUAD_SIZE)]
    for k in range(SQUAD_SIZE):
        bc_obs.append(obs_list[k].copy())
        bc_act.append(actions[k])
    obs_list, rewards, done, info = env.step(actions)
    collected += SQUAD_SIZE
    if done:
        obs_list = env.reset()

bc_obs = np.array(bc_obs[:BC_ROLLOUT_STEPS], dtype=np.float32)
bc_act = np.array(bc_act[:BC_ROLLOUT_STEPS], dtype=np.int64)

print(f'Collected {len(bc_obs):,} pairs in {time.time()-t0:.1f}s')
print(f'  action distribution:')
unique, counts = np.unique(bc_act, return_counts=True)
for u, c in zip(unique, counts):
    print(f'    action {u:2d}: {c:>6,} ({100*c/len(bc_act):.1f}%)')

In [ ]:
# Build the policy net architecture and train via supervised BC
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'BC device: {device}')

# Match SB3 default MlpPolicy: 64x64 → action logits.
# We'll use 128x128 for capacity.
NET_HIDDEN = [128, 128]

class PolicyMLP(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=NET_HIDDEN):
        super().__init__()
        layers = []
        in_dim = obs_dim
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.Tanh()]
            in_dim = h
        self.shared = nn.Sequential(*layers)
        self.action_head = nn.Linear(in_dim, act_dim)
    def forward(self, x):
        return self.action_head(self.shared(x))

bc_model = PolicyMLP(OBS_DIM, ACTION_DIM).to(device)
opt = torch.optim.Adam(bc_model.parameters(), lr=1e-3)

bc_obs_t = torch.from_numpy(bc_obs).to(device)
bc_act_t = torch.from_numpy(bc_act).to(device)

import time; t0 = time.time()
bs = 1024
n = len(bc_obs_t)
for epoch in range(BC_EPOCHS):
    perm = torch.randperm(n, device=device)
    losses = []
    correct = 0
    for i in range(0, n, bs):
        idx = perm[i:i+bs]
        x = bc_obs_t[idx]
        y = bc_act_t[idx]
        logits = bc_model(x)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        correct += (logits.argmax(-1) == y).sum().item()
    if (epoch+1) % max(1, BC_EPOCHS // 10) == 0 or epoch == 0:
        print(f'  BC epoch {epoch+1}/{BC_EPOCHS}: loss={np.mean(losses):.3f} '
              f'acc={100*correct/n:.1f}%  ({time.time()-t0:.1f}s)')
print(f'BC training done in {time.time()-t0:.1f}s')

## Build the vectorized env with curriculum + opponent factory

Each parallel env queries `_global_step.value` at every reset() — that's
how the curriculum advances over time. The trainer's callback bumps
`_global_step.value` on each rollout.

In [ ]:
import multiprocessing as mp
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor

# Shared step counter (one int across all worker processes)
_global_step = mp.Value('i', 0)
_step_total  = mp.Value('i', TOTAL_STEPS)

# Self-play pool — list of paths to .zip snapshots. Workers cache loaded
# policies in a process-local dict (avoids reloading from disk every reset).
_self_pool_paths = mp.Manager().list()
_ga_genome_global = DEFAULT_GA_GENOME

# Per-worker policy cache (each worker process gets its own copy of this dict)
_worker_policy_cache = {}

def _load_self_policy(path):
    if path in _worker_policy_cache:
        return _worker_policy_cache[path]
    try:
        from stable_baselines3 import PPO as _PPO
        m = _PPO.load(path, device='cpu')
        def _policy(obs_list, env, _m=m):
            obs_arr = np.stack(obs_list).astype(np.float32)
            actions, _ = _m.predict(obs_arr, deterministic=False)
            return [int(a) for a in actions]
        _worker_policy_cache[path] = _policy
        return _policy
    except Exception:
        return None

def make_opponent_for_curriculum(c):
    # Build a self-pool of callables (cached per-worker)
    pool_callables = []
    if c.p_self > 0 and len(_self_pool_paths) > 0:
        for p in list(_self_pool_paths)[-SELF_POOL_MAX:]:
            if not os.path.exists(p):
                continue
            pol = _load_self_policy(p)
            if pol is not None:
                pool_callables.append(pol)
    # Drop stale cache entries (removed snapshots)
    current = set(list(_self_pool_paths))
    for stale in [k for k in _worker_policy_cache if k not in current]:
        _worker_policy_cache.pop(stale, None)

    picker = make_curriculum_opponent_picker(
        ga_genome=_ga_genome_global,
        self_pool=pool_callables,
    )
    return picker(c)

def make_env(rank: int):
    def _init():
        # Each worker reads the shared step counter
        seed = 1000 + rank * 7919
        def provider():
            step = _global_step.value
            total = _step_total.value
            return curriculum_for_step(step, total)
        env = SinglePerspectiveEnv(
            curriculum_provider=provider,
            opponent_factory=make_opponent_for_curriculum,
            squad_size=SQUAD_SIZE,
            seed=seed,
        )
        return env
    return _init

# Quick test: one env produces obs + reward
test_env = make_env(0)()
obs0, _ = test_env.reset(seed=42)
print(f'Single env test: obs shape={obs0.shape}, sample_reward over 100 steps:')
total_r = 0
for _ in range(100):
    a = test_env.action_space.sample()
    o, r, d, t, info = test_env.step(a)
    total_r += r
    if d: break
print(f'  reward over 100 steps: {total_r:.2f}')

# Build the vectorized env
print(f'Building SubprocVecEnv with {N_ENVS} workers...')
vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
vec_env = VecMonitor(vec_env)
print('Vec env ready.')

## Build PPO model, transfer BC weights

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CallbackList

# Linear schedule for entropy coefficient: ENT_COEF_INIT → ENT_COEF_FINAL
def linear_schedule(initial, final):
    def f(progress_remaining):
        # progress_remaining starts at 1.0, ends at 0.0
        return final + progress_remaining * (initial - final)
    return f

policy_kwargs = dict(
    net_arch=dict(pi=NET_HIDDEN, vf=NET_HIDDEN),
    activation_fn=nn.Tanh,
)

print('Building PPO model...')
model = PPO(
    policy='MlpPolicy',
    env=vec_env,
    learning_rate=LR_INIT,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=CLIP_RANGE,
    ent_coef=linear_schedule(ENT_COEF_INIT, ENT_COEF_FINAL),
    policy_kwargs=policy_kwargs,
    verbose=1,
    device=device,
)

# Transfer BC weights into the SB3 policy.
# SB3 MlpPolicy structure:
#   policy.mlp_extractor.policy_net (shared encoder for actor)
#   policy.action_net               (logits head)
print('Transferring BC weights into PPO policy...')
sb3_pi  = model.policy.mlp_extractor.policy_net
sb3_act = model.policy.action_net

bc_layers = [m for m in bc_model.shared if isinstance(m, nn.Linear)]
sb3_layers = [m for m in sb3_pi if isinstance(m, nn.Linear)]
print(f'  BC layers: {len(bc_layers)}, SB3 layers: {len(sb3_layers)}')

if len(bc_layers) == len(sb3_layers):
    for bc_l, sb_l in zip(bc_layers, sb3_layers):
        if bc_l.weight.shape == sb_l.weight.shape:
            sb_l.weight.data.copy_(bc_l.weight.data)
            sb_l.bias.data.copy_(bc_l.bias.data)
        else:
            print(f'    skip mismatched shapes: bc={bc_l.weight.shape}, sb3={sb_l.weight.shape}')
    if bc_model.action_head.weight.shape == sb3_act.weight.shape:
        sb3_act.weight.data.copy_(bc_model.action_head.weight.data)
        sb3_act.bias.data.copy_(bc_model.action_head.bias.data)
        print('  Action head transferred.')
    else:
        print(f'  Action head shape mismatch: bc={bc_model.action_head.weight.shape}, '
              f'sb3={sb3_act.weight.shape}  — using random init')
else:
    print('  Layer count mismatch — using random init (BC was for nothing :/)')

# Free BC dataset to reclaim RAM
del bc_obs, bc_act, bc_obs_t, bc_act_t
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

n_params = sum(p.numel() for p in model.policy.parameters())
print(f'PPO model ready. Parameters: {n_params:,}')

## Training callback

- Bumps the shared `_global_step` so workers see curriculum progress
- Snapshots policy to `self_pool` every `FROZEN_OPP_EVERY` steps
- Logs win rates, current curriculum stage
- Saves `.zip` checkpoints every `CHECKPOINT_EVERY`

In [ ]:
class CurriculumSelfPlayCallback(BaseCallback):
    def __init__(self, output_dir: str, verbose=0):
        super().__init__(verbose)
        self.output_dir = output_dir
        self.last_freeze = 0
        self.last_checkpoint = 0
        self.last_log = 0

    def _on_step(self) -> bool:
        # SB3 calls this once per env-step (across all workers)
        n = self.num_timesteps
        # Update shared global step so vec_env reset()s see new curriculum
        _global_step.value = n

        # Periodic frozen-opponent snapshot
        if n - self.last_freeze >= FROZEN_OPP_EVERY:
            self.last_freeze = n
            path = os.path.join(self.output_dir, f'self_snap_{n}.zip')
            try:
                self.model.save(path)
                _self_pool_paths.append(path)
                # Trim oldest if pool too big
                while len(_self_pool_paths) > SELF_POOL_MAX:
                    old = _self_pool_paths.pop(0)
                    try:
                        if os.path.exists(old):
                            os.remove(old)
                    except OSError:
                        pass
                if self.verbose:
                    print(f'[selfplay] snapshot @ step {n:,}: pool size={len(_self_pool_paths)}')
            except Exception as e:
                print(f'[selfplay] snapshot failed: {e}')

        # Periodic checkpoint
        if n - self.last_checkpoint >= CHECKPOINT_EVERY:
            self.last_checkpoint = n
            path = os.path.join(self.output_dir, f'ppo_combat_{n}.zip')
            try:
                self.model.save(path)
                if self.verbose:
                    print(f'[ckpt]  @ step {n:,} -> {path}')
            except Exception as e:
                print(f'[ckpt] save failed: {e}')

        # Log curriculum stage every 100k steps
        if n - self.last_log >= 100_000:
            self.last_log = n
            c = curriculum_for_step(n, TOTAL_STEPS)
            print(f'[curriculum @ {n:>10,}] world={c.world_w}x{c.world_h} '
                  f'spawn={c.spawn_dist:.0f} stat={c.p_static:.2f} ga={c.p_ga:.2f} '
                  f'self={c.p_self:.2f} (pool={len(_self_pool_paths)}) '
                  f'vis={c.coef_visibility:.3f}')

        return True

print('Callback class defined.')

## Train

Now the long-running cell. SB3's progress bar shows `ep_rew_mean` per
rollout. With the curriculum + BC warm-start you should see:

- Steps 0–1M:   `ep_rew_mean` rises from BC level (~3-8) into the +20 range
- Steps 1M–4M:  steady climb into +30..+50 (medium-map combat)
- Steps 4M–8M:  +50..+100 (full deployment scale, self-play)
- Steps 8M–16M: continued refinement, win-rate vs GA-best should pass 70%

If you see `ep_rew_mean` stuck at 0 after 500k steps, something's wrong
with reward shaping or env wiring (check the [curriculum @ ...] log lines).

In [ ]:
import time
t0 = time.time()
print(f'>>> Training for {TOTAL_STEPS:,} steps on {device}...')

cb = CurriculumSelfPlayCallback(output_dir=OUTPUT_DIR, verbose=1)
model.learn(total_timesteps=TOTAL_STEPS, callback=cb, progress_bar=False)

elapsed = time.time() - t0
print(f'>>> Training done in {elapsed/60:.1f} min ({elapsed/3600:.2f} h)')
final_path = os.path.join(OUTPUT_DIR, 'ppo_combat_final.zip')
model.save(final_path)
print(f'Final model saved to {final_path}')

## Export to ONNX

The JS game loads ONNX via `onnxruntime-web`. We wrap the SB3 policy in
a tiny module that exposes `forward(obs) -> action_probs`.

In [ ]:
import torch
import torch.nn as nn

class OnnxablePolicy(nn.Module):
    def __init__(self, sb3_policy):
        super().__init__()
        self.extractor = sb3_policy.mlp_extractor
        self.action_net = sb3_policy.action_net
    def forward(self, obs):
        latent_pi = self.extractor.policy_net(obs)
        logits = self.action_net(latent_pi)
        return torch.softmax(logits, dim=-1)

# Move policy to CPU for export
model.policy.to('cpu')
onnxable = OnnxablePolicy(model.policy).eval()

dummy = torch.zeros(1, OBS_DIM, dtype=torch.float32)
onnx_path = os.path.join(OUTPUT_DIR, 'model.onnx')
torch.onnx.export(
    onnxable, dummy, onnx_path,
    input_names=['obs'], output_names=['action_probs'],
    dynamic_axes={'obs': {0: 'batch'}, 'action_probs': {0: 'batch'}},
    opset_version=17,
    dynamo=False,
)
print(f'ONNX exported to {onnx_path} ({os.path.getsize(onnx_path)/1024:.1f} KB)')

# Verify it loads + runs
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
test_in = np.random.randn(1, OBS_DIM).astype(np.float32)
out = sess.run(None, {'obs': test_in})
print(f'ONNX inference: input {test_in.shape} -> output {out[0].shape}')
print(f'  sample probs: {out[0][0][:6]}... (sum={out[0][0].sum():.3f})')

# Restore policy to GPU for any further training
if torch.cuda.is_available():
    model.policy.to('cuda')

## Sanity check: trained NN vs GA-best

Run 20 deathmatches at deployment scale (1200×1200, 700u spawn, 45s).
Should win ≥ 65% if curriculum worked.

In [ ]:
print('Running 20 NN-vs-GA-best matches at deployment scale...')
deploy_curriculum = curriculum_for_step(TOTAL_STEPS, TOTAL_STEPS)  # stage 4

wins = losses = draws = 0
ep_rewards = []

for match in range(20):
    env = CombatEnv(opponent_policy=ga_policy,
                    curriculum=deploy_curriculum,
                    seed=20000 + match)
    obs_list = env.reset(seed=20000 + match)
    total_reward = 0.0
    for _ in range(env.match_ticks):
        actions = []
        for k in range(SQUAD_SIZE):
            obs = obs_list[k].astype(np.float32)
            action, _ = model.predict(obs, deterministic=True)
            actions.append(int(action))
        obs_list, rewards, done, info = env.step(actions)
        total_reward += sum(rewards) / SQUAD_SIZE
        if done:
            break
    ep_rewards.append(total_reward)
    if env.team_kills[0] > env.team_kills[1]:   wins += 1
    elif env.team_kills[1] > env.team_kills[0]: losses += 1
    else:                                        draws += 1
    if (match+1) % 5 == 0:
        print(f'  match {match+1}/20: kills={env.team_kills}  W/L/D={wins}/{losses}/{draws}')

print(f'\nFinal:  {wins} W / {losses} L / {draws} D')
print(f'Mean episode reward: {np.mean(ep_rewards):+.1f}')
if wins >= 13:
    print('[OK] NN beats GA-best decisively (good).')
elif wins >= 9:
    print('[OK] NN comparable to GA-best (acceptable for first checkpoint).')
else:
    print('[WARN] NN losing to GA-best — try more steps or check reward shaping')

## Output files

In Kaggle's right-side **Output** panel:

- **`model.onnx`** — drop into the AshGrid repo at `ai_arena/onnx/model.onnx`
- **`ppo_combat_final.zip`** — full SB3 model, can resume training
- **`ppo_combat_<step>.zip`** — checkpoint per CHECKPOINT_EVERY (use as
  difficulty tiers: rename `_easy/_medium/_hard` for the JS lobby)

The browser-side wiring (NN inference loop, lobby, deathmatch mission)
is already in `index.html` on the `claude/ai-ppo-nn` branch — just
replace the existing `model.onnx` and reload the page.